# Word LLM Translation Workflow — Primary Translation Stage

- **Workflow stage:** `primary_translation_completed` (written at the end of this notebook)
- **Input checkpoint:** batched element JSON from Notebook 0
- **Word source:** Basics of the Bible Introduction (example Word file)
- **Source language:** loaded from checkpoint metadata
- **Target language:** user-defined in this notebook
- **Primary translator model:** user-defined in this notebook
- **Primary translation engine:** Gemini API

## Purpose of this notebook
This notebook loads the batched checkpoint from the previous stage, applies the full workflow schema, defines the primary translation prompt in-notebook, and runs the primary translation pass over the extracted elements.

## Primary translator model note

In this workflow, the primary translation stage is configured to use a Gemini model. Do not substitute a different provider or model API path here unless you also update the corresponding primary-translation helper functions in `workflow_helpers.py`, since the current implementation is written for the Gemini client and request/response pattern.

## Metadata flow
This notebook:

- reads `source_language` and prior workflow state from checkpoint metadata
- defines and records `target_language`
- defines and records `primary_model_name`
- saves an updated checkpoint with both `metadata` and `elements`

## Notes
- The primary system prompt is defined directly in this notebook so it is easy to inspect and modify.
- The checkpoint format used in this workflow is a top-level JSON dictionary with:
  - `metadata`
  - `elements`

## Import elements from json
### List contents of */checkpoints* subdirectory

In [1]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints)

Found JSON files:
- elements_batched_20260411_1519.json


In [2]:
# Load checkpoint state (metadata + elements)

import os
from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "elements_batched_20260411_1519.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Stage:", metadata.get("stage"))
print("Source language:", metadata.get("source_language"))
print("Target language:", metadata.get("target_language"))
print(f"Loaded checkpoint with {len(elements)} elements.")
print("Example entry:")
elements[0]

Loaded checkpoint: checkpoints\elements_batched_20260411_1519.json
Stage: elements_batched
Source language: English
Target language: None
Loaded checkpoint with 29 elements.
Example entry:


{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1}

In [ ]:
# User-entered values for this stage
target_language = "Traditional Chinese"
gemini_model_name = "gemini-3.1-pro-preview"

In [ ]:
# Languages and primary model for this notebook stage

# Pull source language forward from checkpoint metadata
source_language = metadata.get("source_language")

if not source_language:
    raise ValueError(
        "source_language was not found in checkpoint metadata. "
        "Please verify the upstream checkpoint file."
    )


# Update metadata now that these values are known
metadata["target_language"] = target_language
metadata["primary_model_name"] = gemini_model_name

print("Source language:", source_language)
print("Target language:", target_language)
print("Primary model:", gemini_model_name)

Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview


In [4]:
elements[0]

{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1}

#### Apply schema to elements

In [5]:
# Apply LLM response schema to elements object

import importlib, workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)
elements = workflow_helpers.apply_element_schema(elements)

Applied schema (12 fields) to 29 elements (overwrite_existing=False).


In [6]:
elements[0]

{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1,
 'primary_translation': None,
 'primary_translation_model': None,
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

### Define primary translation system prompt

#### Adapting the primary translation prompt

This prompt is written for the current example document and translation direction, but it is intended to be edited for your own use case.

Best practices when adapting it:

- update the audience, domain, and style expectations so they match your actual document rather than leaving example-specific wording in place
- keep the JSON contract and output-structure instructions stable unless you also update the downstream helper logic that parses the response
- keep formatting-preservation rules aligned with the kinds of markup your workflow actually needs to preserve
- revise domain-specific terminology guidance for your source material (for example, theological, legal, technical, educational, or literary language)
- if you are unsure how to rewrite the prompt, use a strong LLM to help draft a revised system prompt, then review it carefully before using it in the workflow
- after updating the prompt, run a small one-batch smoke test before launching the full translation pass

In general, change the domain-specific translation guidance freely, but be cautious about changing the JSON schema, field names, or strict output rules unless you are also updating the downstream workflow code.

In [7]:
# Primary translation prompt
# Keep this prompt in the notebook so the workflow is easy to inspect and modify.

primary_system_message = f"""
You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.

You will receive a single JSON object with the following structure:

{{
  "elements": [
    {{
      "id": "<string>",
      "text": "<chunked Markdown in {source_language}>"
    }},
    ...
  ]
}}

Each `text` field is a small chunk of Markdown in {source_language}. For each element, you must translate the natural-language prose into {target_language} while preserving formatting and structure.

You must respond with a single valid JSON object of the form:

{{
  "elements": [
    {{
      "id": "<same id value as input>",
      "translated_text": "<translated Markdown in {target_language}>"
    }},
    ...
  ]
}}

REQUIREMENTS

1) JSON contract:
   - Do NOT add any extra top-level keys; only return the object with the single key "elements".
   - "elements" must be an array with the same number of entries as the input.
   - Each output element must contain exactly:
       - "id": the same string as in the corresponding input element.
       - "translated_text": the translated Markdown chunk in {target_language}.
   - Do NOT include the original "text" field in the output.
   - The response must be syntactically valid JSON. No comments, no trailing commas, no explanation, no prose outside the JSON.

2) Input interpretation:
   - Treat each `text` as chunked Markdown. You are NOT changing the structure of the JSON, only the content of `translated_text` for each element.

3) Translation guidelines (Markdown + theology):
   - Translate chunked Markdown from {source_language} into {target_language}, translating only natural-language prose and leaving protected content unchanged.
   - Preserve formatting and structure: headings, lists, blockquotes, tables, horizontal rules, line breaks, and blank lines. Do not reflow paragraphs.
   - Keep code/data literal: inline code, code fences (``` ... ```), math/code spans, JSON, and YAML front matter — copy verbatim.
   - Custom fences: reproduce `::: ...` and the matching closing `:::` character-for-character. Do not translate or reflow anything inside them.
   - Do not translate or alter: placeholder tokens/IDs ({{VAR}}, {{handlebars}}), XML/HTML/SVG ids, filenames/extensions; link destinations in `[](...)`, URLs, and image paths (translate the visible link text only).
   - Numbering and lettering: copy list markers and numbering/lettering verbatim (e.g., "A.\\t", "1.", "(a)"), including tabs/indentation. Do not renumber or change bullet styles.
   - Scripture/theology: use standard {target_language} Protestant usage for book names and theological terms; keep verse numbers/punctuation as given (e.g., 3:13–14). Do not invent references.
   - Consistency: use consistent {target_language} equivalents for key terms across chunks (e.g., justification, sanctification, grace, Holy Spirit).
   - Mixed content: when prose appears alongside protected tokens or markup, translate only the prose and copy protected parts verbatim.
   - Language detection: if the input `text` for an element is already primarily in {target_language}, copy it unchanged into `translated_text`.
   - Safety valve: if translating a span would break markup/formatting, leave that span in {source_language} instead of altering structure.
   - Ordering: preserve the order of elements. The i-th output element must correspond to the i-th input element and carry the same "id".

4) Output discipline:
   - Your entire response must be exactly one JSON object with the described structure.
   - Do not include any explanations, comments, or additional text outside the JSON.
   - Do not wrap the JSON in backticks or a code block.

Remember: you are translating the Markdown *inside* the `text` fields into {target_language}, and returning a parallel `elements` array where each item has the same "id" and a new "translated_text" field.
""".strip()

print("Primary system prompt defined in notebook.")

Primary system prompt defined in notebook.


### Initialize Gemini model

In [8]:
# Load Gemini client/model interface
# This notebook only uses Gemini for the primary translation stage.

import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

gemini_client = workflow_helpers.initialize_gemini_client()

print("Gemini client initialized.")
print("Primary model:", gemini_model_name)

Gemini client initialized successfully: <google.genai.client.Client object at 0x00000299C7A65250>
Gemini client initialized.
Primary model: gemini-3.1-pro-preview


# Primary Translation

#### Sanity check before sending to LLM

In [9]:
import importlib
from collections import Counter
import workflow_helpers

# Reload to pick up latest changes from workflow_helpers.py
workflow_helpers = importlib.reload(workflow_helpers)

# Sanity check: what batch_numbers do we have?
batch_numbers = sorted({el["batch_number"] for el in elements})
print(f"Detected {len(batch_numbers)} batches.")
print("First few batch numbers:", batch_numbers[:10])

# Optional: see how many elements per batch (first few only)
batch_counts = Counter(el["batch_number"] for el in elements)
print("\nExample batch sizes:")
for bn in batch_numbers[:5]:
    print(f"  batch {bn}: {batch_counts[bn]} elements")


Detected 12 batches.
First few batch numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Example batch sizes:
  batch 1: 3 elements
  batch 2: 1 elements
  batch 3: 4 elements
  batch 4: 3 elements
  batch 5: 3 elements


### Run a test batch
- This will not be written to the canonical 'elements' object
- You can inspect the results in the 'test_elements' object

In [10]:
# test a single batch without writing back to the full `elements` list

from importlib import reload
import workflow_helpers
workflow_helpers = reload(workflow_helpers)

test_elements = [el.copy() for el in elements if el.get("batch_number") == 1]

test_elements = workflow_helpers.run_primary_translation(
    elements=test_elements,
    gemini_client=gemini_client,
    primary_system_message=primary_system_message,
    primary_model_name=gemini_model_name,
    verbose=True,
    print_status_every_n_batches=1,
)

test_elements[0]

[15:24:18] Starting primary translation: 1 batches detected.
[15:24:27] Progress: 1/1 batches completed.
[15:24:27] Primary translation complete: 1/1 batches processed in 00:00.


{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1,
 'primary_translation': '簡介',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

In [11]:
print(test_elements[0]["text"])
print(test_elements[0]["primary_translation"])
print(test_elements[0]["primary_translation_model"])
print(test_elements[0]["primary_error"])

Introduction
簡介
gemini-3.1-pro-preview
None


In [12]:
len(test_elements)

3

In [13]:
test_elements[0:4]

[{'element_number': 1,
  'element_type': 'paragraph',
  'word_style': 'h1',
  'text': 'Introduction',
  'element_id': '8633e724fc99',
  'tokens': 1,
  'batch_number': 1,
  'primary_translation': '簡介',
  'primary_translation_model': 'gemini-3.1-pro-preview',
  'primary_error': None,
  'evaluator_ran': None,
  'evaluator_passed': None,
  'evaluator_feedback': None,
  'evaluator_error': None,
  'fallback_translation': None,
  'fallback_translation_model': None,
  'fallback_error': None,
  'final': None,
  'final_model': None},
 {'element_number': 2,
  'element_type': 'paragraph',
  'word_style': 'h2',
  'text': 'Welcome',
  'element_id': '2e6212d6aad5',
  'tokens': 1,
  'batch_number': 1,
  'primary_translation': '歡迎',
  'primary_translation_model': 'gemini-3.1-pro-preview',
  'primary_error': None,
  'evaluator_ran': None,
  'evaluator_passed': None,
  'evaluator_feedback': None,
  'evaluator_error': None,
  'fallback_translation': None,
  'fallback_translation_model': None,
  'fallback_

### Run the entire batch
- Prints status updates when verbose is true
- Default status update frequency is every 20 batches; set desired frequency below (e.g., 1 for after every batch)
- Note that when the LLM returns an error, there is no printed status update, only successful batch completions are printed 

In [14]:
# Run primary translation across all batches

from importlib import reload
import workflow_helpers
workflow_helpers = reload(workflow_helpers)

elements = workflow_helpers.run_primary_translation(
    elements=elements,
    gemini_client=gemini_client,
    primary_system_message=primary_system_message,
    primary_model_name=gemini_model_name,
    verbose=True,
    print_status_every_n_batches=1,
)

[15:24:27] Starting primary translation: 12 batches detected.
[15:24:40] Progress: 1/12 batches completed.
[15:24:58] Progress: 2/12 batches completed.
[15:25:11] Progress: 3/12 batches completed.
[15:25:25] Progress: 4/12 batches completed.
[15:25:43] Progress: 5/12 batches completed.
[15:26:00] Progress: 6/12 batches completed.
[15:26:12] Progress: 7/12 batches completed.
[15:26:28] Progress: 8/12 batches completed.
[15:26:48] Progress: 9/12 batches completed.
[15:26:53] Progress: 10/12 batches completed.
[15:27:15] Progress: 11/12 batches completed.
[15:27:33] Progress: 12/12 batches completed.
[15:27:33] Primary translation complete: 12/12 batches processed in 00:03.


In [15]:
# Quick QA after primary translation run

total_elements = len(elements)
translated_count = sum(el.get("primary_translation") is not None for el in elements)
error_count = sum(el.get("primary_error") is not None for el in elements)

print("Total elements:", total_elements)
print("Translated:", translated_count)
print("Errors:", error_count)

if error_count:
    print("\nExample errors:")
    for el in elements:
        if el.get("primary_error") is not None:
            print(
                {
                    "element_id": el.get("element_id"),
                    "batch_number": el.get("batch_number"),
                    "primary_error": el.get("primary_error"),
                }
            )
            break

Total elements: 29
Translated: 29
Errors: 0


In [16]:
# Inspect a few translated examples

for el in elements[:5]:
    print(
        {
            "element_id": el.get("element_id"),
            "text": el.get("text"),
            "primary_translation": el.get("primary_translation"),
            "primary_error": el.get("primary_error"),
        }
    )

{'element_id': '8633e724fc99', 'text': 'Introduction', 'primary_translation': '簡介', 'primary_error': None}
{'element_id': '2e6212d6aad5', 'text': 'Welcome', 'primary_translation': '歡迎', 'primary_error': None}
{'element_id': '4126aaaf767e', 'text': 'This course is designed for the person that wants to learn what it means to become or be a follower of Jesus. Many people that seek this knowledge realize that the Bible is where they must look, but have been hampered in their quest for one reason or another (e.g. intimidated by the size of the Bible, don’t know where to begin, tried to read the Bible but couldn’t make sense of it, got discouraged, etc.).', 'primary_translation': '本課程專為想要了解成為或作為耶穌跟隨者有何意義的人而設計。許多尋求這方面知識的人明白，他們必須從聖經中尋找答案，但在尋求的過程中卻因各種原因受阻（例如：被聖經的篇幅嚇退、不知從何讀起、試著讀聖經卻看不懂、感到氣餒等）。', 'primary_error': None}
{'element_id': '24ef2f4ac85e', 'text': 'This is a **self-directed study** to assist you in your quest for answers. It is intended to take you rapidly through select books of the Bib

In [17]:
for el in elements[:5]:
    print({
        "text": el["text"],
        "primary_translation": el["primary_translation"],
        "primary_error": el["primary_error"],
    })

{'text': 'Introduction', 'primary_translation': '簡介', 'primary_error': None}
{'text': 'Welcome', 'primary_translation': '歡迎', 'primary_error': None}
{'text': 'This course is designed for the person that wants to learn what it means to become or be a follower of Jesus. Many people that seek this knowledge realize that the Bible is where they must look, but have been hampered in their quest for one reason or another (e.g. intimidated by the size of the Bible, don’t know where to begin, tried to read the Bible but couldn’t make sense of it, got discouraged, etc.).', 'primary_translation': '本課程專為想要了解成為或作為耶穌跟隨者有何意義的人而設計。許多尋求這方面知識的人明白，他們必須從聖經中尋找答案，但在尋求的過程中卻因各種原因受阻（例如：被聖經的篇幅嚇退、不知從何讀起、試著讀聖經卻看不懂、感到氣餒等）。', 'primary_error': None}
{'text': 'This is a **self-directed study** to assist you in your quest for answers. It is intended to take you rapidly through select books of the Bible to give you a **“big picture”** overview of what it means to be or become a follower of Jesus, and to become familiar

#### Update metadata

In [18]:
metadata["stage"] = workflow_helpers.WORKFLOW_STAGES["primary_completed"]
metadata["primary_system_message"] = primary_system_message

## Save intermediate state to json

In [19]:
# save intermediate state to json
from importlib import reload
import workflow_helpers

# Reload the updated helper file
workflow_helpers = reload(workflow_helpers)

# Save the checkpoint with metadata + elements
checkpoint_path = workflow_helpers.save_elements_checkpoint(
    elements=elements,
    base_filename=workflow_helpers.WORKFLOW_STAGES["primary_completed"],
    metadata=metadata,
)

print("Checkpoint saved to:", checkpoint_path)
print("Metadata saved:")
print(metadata)

Checkpoint saved to: checkpoints\primary_translation_completed_20260411_1527.json
Metadata saved:
{'stage': 'primary_translation_completed', 'docxfilename': 'custom_word_styles_example.docx', 'source_language': 'English', 'target_language': 'Traditional Chinese', 'primary_model_name': 'gemini-3.1-pro-preview', 'evaluation_model_name': None, 'fallback_model_name': None, 'primary_system_message': 'You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.\n\nYou will receive a single JSON object with the following structure:\n\n{\n  "elements": [\n    {\n      "id": "<string>",\n      "text": "<chunked Markdown in English>"\n    },\n    ...\n  ]\n}\n\nEach `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.\n\nYou must respond with a single valid JSON object of the form:\n\n{\n  "elements": [\n    {\n      "id"

In [20]:
import json

with open(checkpoint_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level type:", type(data).__name__)
print("Keys:", list(data.keys()) if isinstance(data, dict) else None)
print("Metadata:", data.get("metadata") if isinstance(data, dict) else None)
print("First element:", data["elements"][0] if isinstance(data, dict) and data.get("elements") else None)

Top-level type: dict
Keys: ['metadata', 'elements']
Metadata: {'stage': 'primary_translation_completed', 'docxfilename': 'custom_word_styles_example.docx', 'source_language': 'English', 'target_language': 'Traditional Chinese', 'primary_model_name': 'gemini-3.1-pro-preview', 'evaluation_model_name': None, 'fallback_model_name': None, 'primary_system_message': 'You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.\n\nYou will receive a single JSON object with the following structure:\n\n{\n  "elements": [\n    {\n      "id": "<string>",\n      "text": "<chunked Markdown in English>"\n    },\n    ...\n  ]\n}\n\nEach `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.\n\nYou must respond with a single valid JSON object of the form:\n\n{\n  "elements": [\n    {\n      "id": "<same id value as input>",\n     

### Error report

In [21]:
error_counts = sorted({
    el["batch_number"]
    for el in elements
    if el.get("primary_error")
}
                     )

print(f"number of batches with primary error: {len(error_counts)}") 
print(error_counts)

number of batches with primary error: 0
[]


## Notebook checkpoint and handoff

This notebook completed the primary translation stage of the Word-to-LLM workflow by performing the following steps:

- loaded the batched checkpoint JSON from Notebook 0
- loaded workflow metadata and extracted element content from the checkpoint structure
- applied the full workflow schema to the element package
- pulled `source_language` forward from checkpoint metadata
- defined the `target_language` and primary translator model for this stage
- defined the primary translation system prompt directly in the notebook for visibility and editability
- initialized the Gemini client using the updated helper-based workflow
- ran a one-batch smoke test to verify prompt, payload, and response handling
- executed the full primary translation pass across all batches
- updated workflow metadata to reflect the completed primary translation stage
- saved the resulting intermediate state to a timestamped JSON checkpoint together with workflow metadata

### Output of this notebook
The main output is a timestamped JSON checkpoint containing:

- a top-level `metadata` block
- an `elements` list containing the translated elements and the full workflow schema state

This file is intended to be used as input for the next notebook.

### Metadata saved at this stage
The checkpoint metadata currently records:

- `stage = primary_translation_completed`
- `source_language`
- `target_language`
- `primary_model_name`
- placeholder fields for `evaluation_model_name` and `fallback_model_name`, which remain `None` at this stage

### Scope of this notebook
This notebook focuses on the primary translation phase only.

At this stage:

- the primary translation is written to `primary_translation`
- the primary translator model name is written to `primary_translation_model`
- any primary-stage translation issues are written to `primary_error`
- evaluator, fallback, and final fields remain unprocessed placeholders
- the primary system prompt is notebook-owned rather than embedded inside the helper file
- checkpoint save/load behavior now uses a metadata-aware JSON structure

### Next step
The next notebook will load the primary translation checkpoint, identify any batches with `primary_error`, and allow targeted reruns to repair failed primary translations before evaluation begins.

Go to: `2_rerun_primary_translation_errors.ipynb`

If the checkpoint contains no `primary_error` values (this is likely with shorter Word documents whereas long Word documents tend to have several failures), you can skip the rerun notebook and go directly to: `3_primary_translation_analysis.ipynb`